# Lisper Gemma 4 E4B Merge Only

This notebook does not retrain. It loads the recovered E4B LoRA adapter and uses Unsloth's direct Hub merge path to create the private full-model repo.

Required Kaggle secret: `HF_TOKEN` with read access to the private adapter repo and write access to the full model repo.


In [ ]:
!pip install --quiet --no-cache-dir --index-url https://download.pytorch.org/whl/cu124 torch==2.6.0 torchvision==0.21.0 numpy==2.0.2 pillow==11.3.0
!pip install --quiet --no-cache-dir unsloth unsloth_zoo
!pip install --quiet --no-cache-dir "transformers!=5.0.0,!=5.1.0,<=5.5.0,>=4.51.3" "peft" "accelerate" "bitsandbytes" "huggingface_hub" "hf_transfer"


In [ ]:
import gc
import json
import os
from pathlib import Path

import torch
from huggingface_hub import HfApi, login

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ.setdefault("TORCHDYNAMO_DISABLE", "1")
os.environ.setdefault("UNSLOTH_COMPILE_DISABLE", "1")


def get_hf_token() -> str:
    token = os.environ.get("HF_TOKEN")
    if token:
        return token
    from kaggle_secrets import UserSecretsClient

    token = UserSecretsClient().get_secret("HF_TOKEN")
    if not token:
        raise RuntimeError("Missing Kaggle HF_TOKEN secret.")
    return token


HF_TOKEN = get_hf_token()
ADAPTER_REPO = "thomasjvu/lisper-gemma4-e4b-audio-lora"
FULL_REPO = "thomasjvu/lisper-gemma4-e4b-audio-full"
MAX_SEQ_LENGTH = 2048
MERGED_SAVE_METHOD = "merged_16bit"

login(token=HF_TOKEN)
api = HfApi(token=HF_TOKEN)
api.create_repo(repo_id=FULL_REPO, repo_type="model", private=True, exist_ok=True)

print(
    json.dumps(
        {
            "adapter_repo": ADAPTER_REPO,
            "full_repo": FULL_REPO,
            "cuda_available": torch.cuda.is_available(),
            "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
        },
        indent=2,
    )
)


In [ ]:
import unsloth  # noqa: F401
from unsloth import FastVisionModel

model, processor = FastVisionModel.from_pretrained(
    model_name=ADAPTER_REPO,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    full_finetuning=False,
    token=HF_TOKEN,
)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

push_to_hub_merged = getattr(model, "push_to_hub_merged", None)
if push_to_hub_merged is None:
    raise AttributeError("Loaded Unsloth model does not expose push_to_hub_merged.")

push_to_hub_merged(
    FULL_REPO,
    processor,
    save_method=MERGED_SAVE_METHOD,
    token=HF_TOKEN,
)

print(
    json.dumps(
        {
            "status": "pushed_merged_model",
            "adapter_repo": ADAPTER_REPO,
            "full_repo": FULL_REPO,
            "save_method": MERGED_SAVE_METHOD,
        },
        indent=2,
    )
)
